# 01 · Skill extraction study (synopsis §4.3)

Precision, recall and F1 of the taxonomy-constrained extractor against hand-annotated resumes, with per-method precision and a no-fuzzy ablation. Annotations come from the web tool's database (Placement cell → Annotate); if there are none, the synthetic demo set is used.

In [1]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.precision", 3)

# Use the real processed dataset when it exists, otherwise the synthetic demo.
REAL = ROOT / "placeiq-data/processed/historical.csv"
DATA = REAL if REAL.exists() else ROOT / "placeiq-data/synthetic/historical.csv"
SYNTHETIC = DATA != REAL
print("Data:", DATA.relative_to(ROOT), "(SYNTHETIC - do not report)" if SYNTHETIC else "(real)")

Data: placeiq-data\synthetic\historical.csv (SYNTHETIC - do not report)


In [2]:
from placeiq.db import Annotation, session_factory
from placeiq.extraction.evaluate import evaluate_texts
with session_factory()() as s:
    rows = s.query(Annotation).all()
texts = {a.resume_id: a.text for a in rows}
gold = {a.resume_id: a.gold() for a in rows}
synthetic_ann = any(a.annotator == "synthetic" for a in rows)
print(f"{len(rows)} annotated resumes", "(SYNTHETIC)" if synthetic_ann else "")
res = evaluate_texts(texts, gold)
pd.DataFrame({"micro": res["micro"], "macro over documents": res["macro_over_documents"],
              "without fuzzy (ablation)": res["ablation_without_fuzzy"]}).T

50 annotated resumes (SYNTHETIC)


,precision,recall,f1
micro,0.998,0.984,0.991
macro over documents,0.999,0.987,0.992
without fuzzy (ablation),1.000,0.982,0.991


In [3]:
pd.DataFrame(res["precision_by_method"]).T

,correct,predicted,precision
exact,433.0,433.0,1.0
fuzzy,1.0,2.0,0.5
strict,6.0,6.0,1.0


In [4]:
pd.DataFrame({"false positives": pd.Series(dict(res["top_false_positives"])),
              "false negatives": pd.Series(dict(res["top_false_negatives"]))}).fillna(0).astype(int)

,false positives,false negatives
bootstrap,0,1
eclipse,0,1
fastapi,0,1
fortran,0,1
latex,0,1
swagger,1,1
verilog,0,1


**Acceptance criterion:** micro F1 > 0.80 (Table 1). Report precision and recall separately too: a low-precision extractor inflates skill counts, a low-recall one hides skills from the model.